# Flood-It: Engagement and New-User Retention

**Data.** Google's public Firebase export for the mobile game Flood-It (`firebase-public-project.analytics_153293282`, 2018-06-12 to 2018-10-03). All queries run in BigQuery (`../sql/`); this notebook only reads the small aggregated result files in `../results/`.

**Definitions.**
- *Active* on a date = at least one logged event that day.
- *New installs* = 5,152 users whose first app open falls inside the export window and within one day of their first logged event (see `../sql/02_validate_user_tables.sql`). Day 0 = first logged activity.
- *Day-N retention* = active on exactly day 0 + N. Users whose day N falls after 2018-10-03 are excluded from that metric (right-censoring).

## 1. Load results

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

daily = pd.read_csv("../results/daily_active_users.csv", parse_dates=["date"])
pooled = pd.read_csv("../results/retention_pooled.csv")
monthly = pd.read_csv("../results/retention_by_month.csv")
print(daily.shape, pooled.shape, monthly.shape)

## 2. Daily active users

DAU for all 15,175 users. Seven isolated days in late June / early July roughly double the usual level and return to normal the next day; their cause is not investigated here. MAU (30-day window) is shown from 2018-07-11, the first date with a full look-back.

In [ ]:
import matplotlib.dates as mdates

BLUE, TEXT, MUTED, GRID = "#257de0", "#191919", "#666666", "#e5e5e5"

spikes = daily[daily["dau"] > 1000]
print("spike days:", list(spikes["date"].dt.strftime("%m-%d")))
print("avg DAU:", round(daily["dau"].mean(), 1), "| avg MAU (full window):", round(daily["mau"].mean(), 1),
      "| avg DAU/MAU:", round(daily["dau_mau"].mean(), 4))

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 5), sharex=True, gridspec_kw={"height_ratios": [3, 2]})
ax1.plot(daily["date"], daily["dau"], color=BLUE, linewidth=2)
ax1.plot(spikes["date"], spikes["dau"], "o", color=BLUE, markersize=8, markeredgecolor="white", markeredgewidth=2)
ax1.annotate("7 one-day spikes (cause not investigated)", xy=(spikes["date"].iloc[-1], spikes["dau"].iloc[-1]),
             xytext=(15, -4), textcoords="offset points", color=TEXT, fontsize=9, va="center")
ax1.set_ylabel("DAU", color=MUTED)
ax1.set_title("Daily active users were stable at about 450–550 outside seven spike days",
              color=TEXT, loc="left", fontsize=11)

ax2.plot(daily["date"], daily["mau"], color=BLUE, linewidth=2)
ax2.set_ylabel("MAU (30-day)", color=MUTED)

for ax in (ax1, ax2):
    ax.grid(axis="y", color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ["top", "right", "left"]:
        ax.spines[side].set_visible(False)
    ax.tick_params(colors=MUTED, length=0)
    ax.set_ylim(bottom=0)
ax2.xaxis.set_major_locator(mdates.MonthLocator())
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))

fig.text(0.01, -0.03, "All 15,175 users. Active = at least one logged event that day. "
         "MAU starts on 2018-07-11, the first date with a full 30-day window.", color=MUTED, fontsize=8)
fig.tight_layout()
fig.savefig("../results/fig3_daily_active_users.png", dpi=200, bbox_inches="tight")
plt.show()

## 3. Retention curves by install month

Pooled across all new installs: D1 21.5%, D7 5.5%, D14 3.4%, D30 1.9% (`../results/retention_pooled.csv`). By install month, the September cohort retained clearly worse on day 1 and day 7. September D30 is omitted because only 173 September users can be observed for 30 days.

In [ ]:
COLORS = {"2018-06": "#257de0", "2018-07": "#f47230", "2018-08": "#00b894", "2018-09": "#f2a600"}
LABELS = {"2018-06": "June", "2018-07": "July", "2018-08": "August", "2018-09": "September"}

plot = monthly[~((monthly["cohort_month"] == "2018-09") & (monthly["day_n"] == 30))]

fig, ax = plt.subplots(figsize=(8, 4.2))
for month, grp in plot.groupby("cohort_month"):
    ax.plot(grp["day_n"], grp["retention_rate"] * 100, color=COLORS[month], linewidth=2,
            marker="o", markersize=8, markeredgecolor="white", markeredgewidth=2, label=LABELS[month])

d1 = monthly[monthly["day_n"] == 1].set_index("cohort_month")["retention_rate"] * 100
ax.annotate(f"September D1: {d1['2018-09']:.1f}% vs {d1['2018-08']:.1f}% in August",
            xy=(1, d1["2018-09"]), xytext=(10, 17), textcoords="data", color=TEXT, fontsize=9,
            arrowprops=dict(arrowstyle="-", color=MUTED, linewidth=1))

ax.set_xticks([1, 7, 14, 30])
ax.set_xlim(0, 35)
ax.set_ylim(0, 30)
ax.set_xlabel("Days since first activity", color=MUTED)
ax.set_ylabel("Day-N retention (%)", color=MUTED)
ax.set_title("New users from September came back less often", color=TEXT, loc="left", fontsize=11)
ax.legend(frameon=False, loc="upper right", labelcolor=TEXT)
ax.grid(axis="y", color=GRID, linewidth=0.8)
ax.set_axisbelow(True)
for side in ["top", "right", "left"]:
    ax.spines[side].set_visible(False)
ax.tick_params(colors=MUTED, length=0)

fig.text(0.01, -0.06, "New installs only (5,152 users), by install month. Users are excluded from day N if day N falls "
         "after 2018-10-03. June D1 is inflated by spike days (22.7% for June users whose day 1 was not a spike day).",
         color=MUTED, fontsize=8, wrap=True)
fig.tight_layout()
fig.savefig("../results/fig4_retention_by_month.png", dpi=200, bbox_inches="tight")
plt.show()